# Multi-Asset Portfolio Analytics

Analyze performance, risk, and rebalancing for a weighted portfolio against a benchmark. The notebook exports an interactive HTML report and CSV tables.


## 1. Environment setup

Run the install cell in Colab. If you installed the dependencies locally, skip it. Open this notebook from the repository root or the notebooks directory.


In [ ]:
from pathlib import Path

requirements = Path("requirements.txt")
if not requirements.is_file():
    requirements = Path("../requirements.txt")
%pip install -q -r {requirements}


In [ ]:
from pathlib import Path
import sys
import logging
import json

import numpy as np
import pandas as pd

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(
        f"Could not find the src folder. Current directory: {CURRENT_DIR}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

pd.options.display.float_format = "{:,.6f}".format
np.random.seed(42)

print("Notebook directory:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())

## 2. Imports and validated configuration

In [ ]:
from src.config import PortfolioConfig
from src.data_loader import download_prices
from src.data_validator import validate_and_align_prices
from src.returns import simple_returns, log_returns, portfolio_returns, wealth_index
from src.performance import summary, drawdown_series
from src.benchmark import benchmark_summary, capm_regression
from src.risk import risk_summary
from src.attribution import return_contributions, risk_contributions, diversification_ratio
from src.rolling import rolling_metrics
from src.stress_testing import historical_stress_tests, hypothetical_shock
from src.rebalancing import simulate_rebalancing
from src.visualization import (
    growth_chart, drawdown_chart, rolling_chart,
    risk_contribution_chart, stress_chart
)
from src.reporting import export_tables, export_figures_html
from src.optimization import (
    estimate_expected_returns,
    estimate_covariance_matrix,
    minimum_variance_portfolio,
    maximum_sharpe_portfolio,
    efficient_frontier,
)


In [ ]:
config = PortfolioConfig(
    weights={
        "AAPL": 0.20,
        "MSFT": 0.20,
        "SPY": 0.25,
        "TLT": 0.15,
        "GLD": 0.10,
        "BTC-USD": 0.10,
    },
    benchmark="SPY",
    start_date="2018-01-01",
    end_date="2026-07-01",
    initial_capital=100_000,
    annual_risk_free_rate=0.04,
    confidence_level=0.95,
    rolling_short=63,
    rolling_long=126,
    rebalancing_frequency="ME",
    transaction_cost_bps=5.0,
)
weights = pd.Series(config.weights, name="Weight")
config

## 3. Market-data download and caching

In [ ]:
all_tickers = sorted(set(config.tickers) | {config.benchmark})
all_prices = download_prices(
    all_tickers,
    start=config.start_date,
    end=config.end_date,
    cache_dir=PROJECT_ROOT / "data/cache",
    refresh=False,
)
display(all_prices.tail())

## 4. Validation, cleaning and alignment

In [ ]:
asset_raw = all_prices[list(config.tickers)]
benchmark_raw = all_prices[config.benchmark]
asset_prices, benchmark_prices, data_quality = validate_and_align_prices(
    asset_raw, benchmark_raw
)
display(data_quality)
print("Aligned observations:", len(asset_prices))

## 5. Returns, wealth and active performance

In [ ]:
asset_returns = simple_returns(asset_prices).dropna()
asset_log_returns = log_returns(asset_prices).dropna()
benchmark_returns = simple_returns(benchmark_prices).dropna().rename("Benchmark")

aligned = asset_returns.join(benchmark_returns, how="inner").dropna()
asset_returns = aligned[list(config.tickers)]
benchmark_returns = aligned["Benchmark"]
portfolio_daily = portfolio_returns(asset_returns, weights)

portfolio_wealth = wealth_index(portfolio_daily, config.initial_capital)
benchmark_wealth = wealth_index(benchmark_returns, config.initial_capital)

display(pd.concat([portfolio_daily, benchmark_returns], axis=1).head())

## 6. Performance and benchmark summary

In [ ]:
portfolio_performance = summary(portfolio_daily, config.annual_risk_free_rate)
benchmark_performance = summary(benchmark_returns, config.annual_risk_free_rate)
performance_table = pd.concat(
    [portfolio_performance.rename("Portfolio"), benchmark_performance.rename("Benchmark")],
    axis=1,
)
benchmark_metrics = benchmark_summary(
    portfolio_daily, benchmark_returns, config.annual_risk_free_rate
)
display(performance_table)
display(benchmark_metrics.to_frame("Value"))

In [ ]:
capm_model = capm_regression(
    portfolio_daily, benchmark_returns, config.annual_risk_free_rate
)
print(capm_model.summary())

## 7. Tail-risk analytics

In [ ]:
portfolio_risk = risk_summary(portfolio_daily, config.confidence_level)
benchmark_risk = risk_summary(benchmark_returns, config.confidence_level)
risk_table = pd.concat(
    [portfolio_risk.rename("Portfolio"), benchmark_risk.rename("Benchmark")],
    axis=1,
)
display(risk_table)

## 8. Return and risk attribution

In [ ]:
daily_contributions = return_contributions(asset_returns, weights)
risk_attribution = risk_contributions(asset_returns, weights)
risk_attribution["Approx. Annual Return Contribution"] = daily_contributions.mean() * 252
div_ratio = diversification_ratio(asset_returns, weights)

display(risk_attribution)
print(f"Diversification ratio: {div_ratio:.3f}")

## 9. Portfolio optimization and efficient frontier


In [ ]:
expected_returns = estimate_expected_returns(asset_returns)
covariance_matrix = estimate_covariance_matrix(asset_returns)

minimum_variance = minimum_variance_portfolio(
    expected_returns=expected_returns,
    covariance_matrix=covariance_matrix,
    risk_free_rate=config.annual_risk_free_rate,
    max_weight=0.40,
)

maximum_sharpe = maximum_sharpe_portfolio(
    expected_returns=expected_returns,
    covariance_matrix=covariance_matrix,
    risk_free_rate=config.annual_risk_free_rate,
    max_weight=0.40,
)

frontier = efficient_frontier(
    expected_returns=expected_returns,
    covariance_matrix=covariance_matrix,
    points=50,
    max_weight=0.40,
)

In [ ]:
current_return = float(weights @ expected_returns)

current_volatility = float(
    np.sqrt(
        weights.T
        @ covariance_matrix
        @ weights
    )
)

current_sharpe = (
    current_return - config.annual_risk_free_rate
) / current_volatility

optimization_summary = pd.DataFrame(
    {
        "Current Portfolio": {
            "Expected Return": current_return,
            "Volatility": current_volatility,
            "Sharpe Ratio": current_sharpe,
        },
        "Minimum Variance": {
            "Expected Return": minimum_variance.expected_return,
            "Volatility": minimum_variance.volatility,
            "Sharpe Ratio": minimum_variance.sharpe_ratio,
        },
        "Maximum Sharpe": {
            "Expected Return": maximum_sharpe.expected_return,
            "Volatility": maximum_sharpe.volatility,
            "Sharpe Ratio": maximum_sharpe.sharpe_ratio,
        },
    }
).T

display(optimization_summary.style.format({
    "Expected Return": "{:.2%}",
    "Volatility": "{:.2%}",
    "Sharpe Ratio": "{:.2f}",
}))

print("Minimum-Variance Weights")
display(minimum_variance.weights.to_frame().style.format("{:.2%}"))

print("Maximum-Sharpe Weights")
display(maximum_sharpe.weights.to_frame().style.format("{:.2%}"))

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

frontier_figure = px.line(
    frontier,
    x="Volatility",
    y="Expected Return",
    title="Efficient Frontier",
    labels={
        "Volatility": "Annualized Volatility",
        "Expected Return": "Annualized Expected Return",
    },
)

frontier_figure.add_trace(
    go.Scatter(
        x=[minimum_variance.volatility],
        y=[minimum_variance.expected_return],
        mode="markers",
        marker={"size": 14},
        name="Minimum Variance",
    )
)

frontier_figure.add_trace(
    go.Scatter(
        x=[maximum_sharpe.volatility],
        y=[maximum_sharpe.expected_return],
        mode="markers",
        marker={"size": 16, "symbol": "star"},
        name="Maximum Sharpe",
    )
)

frontier_figure.add_trace(
    go.Scatter(
        x=[current_volatility],
        y=[current_return],
        mode="markers",
        marker={"size": 14, "symbol": "diamond"},
        name="Current Portfolio",
    )
)

frontier_figure.update_layout(
    xaxis_tickformat=".1%",
    yaxis_tickformat=".1%",
    hovermode="closest",
)


## 10. Rolling analytics


In [ ]:
rolling = rolling_metrics(
    portfolio_daily,
    benchmark_returns,
    annual_rf=config.annual_risk_free_rate,
    short_window=config.rolling_short,
    long_window=config.rolling_long,
)
display(rolling.tail())

## 11. Historical and hypothetical stress tests

In [ ]:
stress = historical_stress_tests(portfolio_daily, benchmark_returns)
display(stress)

In [ ]:
shocks = {
    "AAPL": -0.30,
    "MSFT": -0.30,
    "SPY": -0.20,
    "TLT": -0.12,
    "GLD": 0.08,
    "BTC-USD": -0.45,
}
hypothetical = hypothetical_shock(weights, shocks)
display(hypothetical)
print("Estimated one-period portfolio shock:",
      f"{hypothetical.loc['TOTAL', 'Contribution']:.2%}")

## 12. Rebalancing and transaction-cost comparison

In [ ]:
buy_hold_returns, buy_hold_costs = simulate_rebalancing(
    asset_returns, weights, frequency=None,
    transaction_cost_bps=0,
    initial_capital=config.initial_capital,
)
monthly_returns, monthly_costs = simulate_rebalancing(
    asset_returns, weights, frequency="ME",
    transaction_cost_bps=config.transaction_cost_bps,
    initial_capital=config.initial_capital,
)
quarterly_returns, quarterly_costs = simulate_rebalancing(
    asset_returns, weights, frequency="QE",
    transaction_cost_bps=config.transaction_cost_bps,
    initial_capital=config.initial_capital,
)

rebalancing_comparison = pd.concat({
    "Buy and Hold": summary(buy_hold_returns, config.annual_risk_free_rate),
    "Monthly": summary(monthly_returns, config.annual_risk_free_rate),
    "Quarterly": summary(quarterly_returns, config.annual_risk_free_rate),
}, axis=1)
display(rebalancing_comparison)
print("Monthly transaction costs:", monthly_costs["Cost"].sum())
print("Quarterly transaction costs:", quarterly_costs["Cost"].sum())

## 13. Interactive visualizations

In [ ]:
figures = {
    "Growth of Capital": growth_chart(portfolio_wealth, benchmark_wealth),
    "Drawdowns": drawdown_chart(
        drawdown_series(portfolio_daily),
        drawdown_series(benchmark_returns)
    ),
    "Rolling Risk": rolling_chart(
        rolling, ["Rolling Volatility", "Rolling VaR 95%"],
        "Rolling Volatility and 95% VaR"
    ),
    "Rolling Benchmark Exposure": rolling_chart(
        rolling, ["Rolling Beta", "Rolling Correlation"],
        "Rolling Beta and Benchmark Correlation"
    ),
    "Risk Contribution": risk_contribution_chart(risk_attribution),
}
if not stress.empty:
    figures["Stress Tests"] = stress_chart(stress)

figures["Efficient Frontier"] = frontier_figure

for figure in figures.values():
    figure.show()

## 14. Executive summary

In [ ]:
def pct(x):
    return "N/A" if pd.isna(x) else f"{x:.2%}"

executive_summary = f"""
PORTFOLIO EXECUTIVE SUMMARY
---------------------------
Analysis period: {portfolio_daily.index.min().date()} to {portfolio_daily.index.max().date()}
Assets: {', '.join(config.tickers)}
Benchmark: {config.benchmark}

Annualized return: {pct(portfolio_performance['Annualized Return'])}
Annualized volatility: {pct(portfolio_performance['Annualized Volatility'])}
Sharpe ratio: {portfolio_performance['Sharpe Ratio']:.2f}
Maximum drawdown: {pct(portfolio_performance['Maximum Drawdown'])}
CAPM beta: {benchmark_metrics['Beta']:.2f}
Annualized Jensen alpha: {pct(benchmark_metrics['Annualized Jensen Alpha'])}
Historical {config.confidence_level:.0%} daily VaR: {pct(portfolio_risk[f'Historical VaR {config.confidence_level:.0%}'])}
Historical {config.confidence_level:.0%} daily CVaR: {pct(portfolio_risk[f'Historical CVaR {config.confidence_level:.0%}'])}
Diversification ratio: {div_ratio:.2f}

Interpretation should consider benchmark suitability, sample-period dependence,
data quality, rebalancing assumptions and omitted transaction/market-impact costs.
"""
print(executive_summary)

## 15. Export tables and interactive HTML report

In [ ]:
tables = {
    "performance_summary": performance_table,
    "benchmark_metrics": benchmark_metrics,
    "risk_summary": risk_table,
    "risk_attribution": risk_attribution,
    "stress_tests": stress,
    "hypothetical_shock": hypothetical,
    "rebalancing_comparison": rebalancing_comparison,
    "data_quality": data_quality,
}
exported = export_tables(tables, PROJECT_ROOT / "reports/exports")
report_path = export_figures_html(
    figures, PROJECT_ROOT / "reports/portfolio_analytics_report.html"
)
print("Exported CSV files:")
for path in exported:
    print("-", path)
print("Interactive HTML report:", report_path)

## 16. Unit tests

In [ ]:
!python -m pytest -q "{PROJECT_ROOT / 'tests'}"


## 17. Assumptions and limitations

- Yahoo Finance is suitable for education and prototyping, not a guaranteed point-in-time institutional feed.
- Current ticker selection creates potential survivorship bias.
- The default portfolio is long-only and uses close-to-close returns.
- The base model does not estimate bid–ask spread, liquidity constraints, taxes or nonlinear market impact.
- Historical metrics are sample-dependent and are not forecasts.
- Parametric VaR assumes a fitted distribution that may understate real tail risk.
- CAPM alpha can reflect omitted factors rather than manager skill.
- Cryptocurrency trades seven days per week; calendar alignment can create zero-return assumptions on non-equity trading days.